In [10]:
from presyo_ph.ml.config import ForecastConfig
from presyo_ph.ml.forecasting import prepare_series
from presyo_ph.medallion.gold import load_gold_table
from presyo_ph.spark import get_spark_session
from pyspark.sql import functions as F

In [11]:
spark = get_spark_session()
gold_df = load_gold_table(
    spark,
    "national_monthly_prices",
)

Gold table loaded: national_monthly_prices (local)


In [12]:
import pandas as pd


def split_monthly_segments(
    pdf: pd.DataFrame,
    date_col: str = "price_date",
) -> list[pd.DataFrame]:

    pdf = pdf.copy()

    pdf[date_col] = pd.to_datetime(pdf[date_col])

    # Normalize dates to the first day of each month
    pdf[date_col] = (
        pdf[date_col]
        .dt.to_period("M")
        .dt.to_timestamp()
    )

    pdf = pdf.sort_values(date_col).reset_index(drop=True)

    # Ensure one observation per month
    if pdf[date_col].duplicated().any():
        raise ValueError("Duplicate monthly observations found.")

    # Convert dates into sequential month numbers
    month_numbers = (
        pdf[date_col].dt.year * 12
        + pdf[date_col].dt.month
    )

    # Start a new segment whenever a month is missing
    segment_ids = month_numbers.diff().ne(1).cumsum()

    segments = [
        group.reset_index(drop=True)
        for _, group in pdf.groupby(segment_ids)
    ]

    return segments

In [13]:
config = ForecastConfig(
    date_col="price_date",
    target_col="avg_price",
    horizon=12,
)

In [ ]:
from presyo_ph.ml.forecasting import (
    prepare_series,
    generate_forecast,
    save_forecasts,
)


MIN_OBSERVATIONS = 36

# Retrieve available commodities
commodities = [
    row["commodity"]
    for row in (
        gold_df
        .select("commodity")
        .distinct()
        .orderBy("commodity")
        .collect()
    )
]

forecast_results = []

for commodity in commodities:

    print(f"\nProcessing: {commodity}")

    pdf = (
        gold_df
        .filter(F.col("commodity") == commodity)
        .select("price_date", "avg_price")
        .orderBy("price_date")
        .toPandas()
    )

    segments = split_monthly_segments(pdf)

    # Select the longest continuous segment
    best_segment = max(
        segments,
        key=len,
    )

    print(
        f"Selected period: "
        f"{best_segment['price_date'].min()} to "
        f"{best_segment['price_date'].max()}"
    )

    print(f"Observations: {len(best_segment)}")

    # Skip segments that are too short
    if len(best_segment) < MIN_OBSERVATIONS:
        print(f"Skipping {commodity}: insufficient observations.")
        continue

    # Prepare the time series
    series = prepare_series(
        best_segment,
        config,
    )

    # Generate forecasts
    forecast_df = generate_forecast(
        series,
        config,
        commodity=commodity,
    )

    # Save to commodity-specific forecast table
    save_forecasts(
        spark,
        forecast_df,
        commodity_name=commodity,
    )

    forecast_results.append({
        "commodity": commodity,
        "observations": len(best_segment),
        "start_date": best_segment["price_date"].min(),
        "end_date": best_segment["price_date"].max(),
        "status": "success",
    })

    print(f"Successfully forecasted {commodity}.")

In [16]:
from presyo_ph.medallion.gold import load_gold_table

forecast_tables = {}

for commodity in commodities:
    forecast_tables[commodity] = (
        load_gold_table(
            spark,
            f"{commodity}_forecasts",
        )
        .withColumn(
            "generated_at",
            F.date_format("generated_at", "yyyy-MM-dd HH:mm:ss"),
        )
        .orderBy("forecast_date")
        .toPandas()
    )

Gold table loaded: beans_forecasts (local)
Gold table loaded: cabbage_forecasts (local)
Gold table loaded: carrots_forecasts (local)
Gold table loaded: eggs_forecasts (local)
Gold table loaded: garlic_forecasts (local)
Gold table loaded: onions_forecasts (local)
Gold table loaded: potatoes_forecasts (local)
Gold table loaded: rice_forecasts (local)
Gold table loaded: tomatoes_forecasts (local)


In [17]:
for commodity in commodities:
    print(f"\nForecasts: {commodity}")

    try:
        forecasts = load_gold_table(
            spark,
            f"{commodity}_forecasts",
        )

        forecasts = forecasts.withColumn(
            "generated_at",
            F.date_format(
                "generated_at",
                "yyyy-MM-dd HH:mm:ss",
            )
        )

        display(
            forecasts
            .orderBy(F.col("forecast_date").asc())
            .limit(5)
            .toPandas()
        )

    except Exception as e:
        print(f"Skipping {commodity}: {e}")


Forecasts: beans
Gold table loaded: beans_forecasts (local)


,forecast_date,commodity,model_name,forecast,lower_bound,upper_bound,generated_at
0,2025-10-01,beans,"ARIMA(1, 1, 1)",112.461004,110.991634,113.930374,2026-11-08 15:10:24
1,2025-11-01,beans,"ARIMA(1, 1, 1)",112.670919,110.254343,115.087496,2026-11-08 15:10:24
2,2025-12-01,beans,"ARIMA(1, 1, 1)",112.753033,109.555269,115.950796,2026-11-08 15:10:24
3,2026-01-01,beans,"ARIMA(1, 1, 1)",112.785153,108.924565,116.645741,2026-11-08 15:10:24
4,2026-02-01,beans,"ARIMA(1, 1, 1)",112.797718,108.359292,117.236143,2026-11-08 15:10:24



Forecasts: cabbage
Gold table loaded: cabbage_forecasts (local)


,forecast_date,commodity,model_name,forecast,lower_bound,upper_bound,generated_at
0,2025-10-01,cabbage,"ARIMA(1, 1, 1)",97.650976,86.856769,108.445183,2026-11-08 15:10:27
1,2025-11-01,cabbage,"ARIMA(1, 1, 1)",94.031199,70.836566,117.225832,2026-11-08 15:10:27
2,2025-12-01,cabbage,"ARIMA(1, 1, 1)",92.866154,59.730889,126.001419,2026-11-08 15:10:27
3,2026-01-01,cabbage,"ARIMA(1, 1, 1)",92.491178,51.179581,133.802775,2026-11-08 15:10:27
4,2026-02-01,cabbage,"ARIMA(1, 1, 1)",92.370490,44.085165,140.655815,2026-11-08 15:10:27



Forecasts: carrots
Gold table loaded: carrots_forecasts (local)


,forecast_date,commodity,model_name,forecast,lower_bound,upper_bound,generated_at
0,2025-10-01,carrots,"ARIMA(1, 1, 1)",183.684652,169.502721,197.866584,2026-11-08 15:10:29
1,2025-11-01,carrots,"ARIMA(1, 1, 1)",185.108926,154.891256,215.326597,2026-11-08 15:10:29
2,2025-12-01,carrots,"ARIMA(1, 1, 1)",185.477771,142.953247,228.002295,2026-11-08 15:10:29
3,2026-01-01,carrots,"ARIMA(1, 1, 1)",185.573290,133.090576,238.056005,2026-11-08 15:10:29
4,2026-02-01,carrots,"ARIMA(1, 1, 1)",185.598027,124.655930,246.540124,2026-11-08 15:10:29


... (8 outputs left out)


Forecasts: rice
Gold table loaded: rice_forecasts (local)


,forecast_date,commodity,model_name,forecast,lower_bound,upper_bound,generated_at
0,2025-10-01,rice,"ARIMA(1, 1, 1)",39.464999,37.589844,41.340154,2026-11-08 15:10:37
1,2025-11-01,rice,"ARIMA(1, 1, 1)",39.410782,36.329008,42.492555,2026-11-08 15:10:37
2,2025-12-01,rice,"ARIMA(1, 1, 1)",39.387512,35.296551,43.478474,2026-11-08 15:10:37
3,2026-01-01,rice,"ARIMA(1, 1, 1)",39.377526,34.422622,44.332430,2026-11-08 15:10:37
4,2026-02-01,rice,"ARIMA(1, 1, 1)",39.373240,33.661815,45.084664,2026-11-08 15:10:37



Forecasts: tomatoes
Gold table loaded: tomatoes_forecasts (local)


,forecast_date,commodity,model_name,forecast,lower_bound,upper_bound,generated_at
0,2025-10-01,tomatoes,"ARIMA(1, 1, 1)",99.515303,64.582869,134.447736,2026-11-08 15:10:39
1,2025-11-01,tomatoes,"ARIMA(1, 1, 1)",98.813087,41.178953,156.447221,2026-11-08 15:10:39
2,2025-12-01,tomatoes,"ARIMA(1, 1, 1)",98.896652,26.055879,171.737425,2026-11-08 15:10:39
3,2026-01-01,tomatoes,"ARIMA(1, 1, 1)",98.886708,13.425318,184.348098,2026-11-08 15:10:39
4,2026-02-01,tomatoes,"ARIMA(1, 1, 1)",98.887891,2.452021,195.323761,2026-11-08 15:10:39


In [29]:
forecasts.printSchema()

root
 |-- forecast_date: date (nullable = true)
 |-- commodity: string (nullable = true)
 |-- model_name: string (nullable = true)
 |-- forecast: double (nullable = true)
 |-- lower_bound: double (nullable = true)
 |-- upper_bound: double (nullable = true)
 |-- generated_at: string (nullable = true)

